In [ ]:
# Prepare the raw GPS table.
import pandas as pd 
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import matplotlib.image as mpimg
import geopandas as gpd 
 
hmm_df = pd.read_csv("hmm_df_b_R_g.csv")

np.random.seed(0) # Set seed for reproducibility.

hmm_df = hmm_df.drop('Time', axis=1)
hmm_df = hmm_df.drop('Pasture_ID', axis=1)
hmm_df = hmm_df.drop('Message_Type', axis=1)
hmm_df = hmm_df.drop('HDOP', axis=1)
hmm_df = hmm_df.drop('Horizontal_Accuracy', axis=1)
hmm_df = hmm_df.drop('n_Satellites', axis=1)
hmm_df = hmm_df.drop('Fence_Status', axis=1)
hmm_df = hmm_df.drop('Collar_Status', axis=1)
hmm_df = hmm_df.drop('day_of_year', axis=1)
hmm_df = hmm_df.drop('gap_seconds', axis=1)
hmm_df = hmm_df.drop('is_15min_target', axis=1)
hmm_df = hmm_df.drop('is_10_20min_gap', axis=1)
hmm_df = hmm_df.drop('run_id', axis=1)
hmm_df = hmm_df.drop('day_part', axis=1)
hmm_df = hmm_df.drop('log_step', axis=1)
hmm_df = hmm_df.drop('log_speed', axis=1)

hmm_df

In [ ]:
state_labels = { 
    1: "Resting",
    2: "Intensive Foraging",
    3: "Extensive Foraging",
    4: "Transit",
}

hmm_df["behaviour_state"] = hmm_df["assigned_state"].map(state_labels)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Set global publication typography
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

custom_colors = {
    "Resting": "grey",            # Matte Dark Grey / Charcoal
    "Intensive Foraging": "lightgreen",   # Crisp Light Yellowish Green
    "Extensive Foraging": "darkgreen",   # Standard Publication Green (ColorBrewer standard)
    "Transit": "orange"               # High-Contrast Pop Orange
}

# 3. Create the clean speed distribution plot
fig, ax = plt.subplots(figsize=(5, 4), dpi=300)

sns.kdeplot(
    data=hmm_df, x="speed_m_per_min", hue="behaviour_state", 
    fill=True, common_norm=False, 
    palette=custom_colors,   
    alpha=0.35,              # Good transparency level for 4 overlapping states
    linewidth=1.2,           
    cut=0,                   # CRITICAL FOR SPEED: Prevents the curve from smoothing into negative speeds
    ax=ax
)

# 4. Enforce strict physical boundary (Speed cannot drop below 0)
ax.set_xlim(left=0)

# 5. Editorial labels and cleanup (Matches the scientific notation from your R code)
ax.set_title("Speed Distribution by Behaviour State", fontsize=11, fontweight='bold', pad=10, loc='left')
ax.set_xlabel(r"Speed (m min$^{-1}$)", fontsize=10, labelpad=6) # Formal scientific notation superscript
ax.set_ylabel("Density", fontsize=10, labelpad=6)

ax.tick_params(labelsize=9)
sns.despine(trim=True)       # Elegant, trimmed L-frame ending exactly at data limits
plt.tight_layout()

plt.savefig("velocity_b_kde.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Set global publication typography
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

custom_colors = {
    "Resting": "grey",            # Matte Dark Grey / Charcoal
    "Intensive Foraging": "lightgreen",   # Crisp Light Yellowish Green
    "Extensive Foraging": "darkgreen",   # Standard Publication Green (ColorBrewer standard)
    "Transit": "orange"               # High-Contrast Pop Orange
}

# 3. Create the clean plot
fig, ax = plt.subplots(figsize=(5, 4), dpi=300)

sns.kdeplot(
    data=hmm_df, x="turning_angle", hue="behaviour_state", 
    fill=True, common_norm=False, 
    palette=custom_colors,   # Manual color assignment
    alpha=0.4,               # Muted transparency so overlaps are clear
    linewidth=1.5,           # Adding a crisp outline defines the shape better
    ax=ax
)

# 4. Clean layout and elegant axis formatting
ax.set_title("Empirical Turning Angles by Behaviour State", fontsize=11, fontweight='bold', pad=10, loc='left')
ax.set_xlabel(r"Turning Angle ($\alpha$, rad)", fontsize=10, labelpad=6) # Math notation matches R
ax.set_ylabel("Density", fontsize=10, labelpad=6)

ax.tick_params(labelsize=9)
sns.despine(trim=True)       # Elegant, trimmed L-frame matching your first figure
plt.tight_layout()

plt.savefig("degree_b_kde.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
from shapely.geometry import Point

# 2. Convert to a GeoDataFrame (Set your project's EPSG code)
geometry = [Point(xy) for xy in zip(hmm_df['x'], hmm_df['y'])]
points_gdf = gpd.GeoDataFrame(hmm_df, geometry=geometry, crs="EPSG:27700")

# 3. Export to a GeoPackage (.gpkg) file for QGIS
# GeoPackages are preferred over Shapefiles as they support longer column names
points_gdf.to_file("qgis_cattle.gpkg", layer="telemetry_fixes", driver="GPKG")
print("Spatial file successfully created for QGIS!")

In [ ]:
resting_df = hmm_df[hmm_df["behaviour_state"] == "Resting"]

# 2. Convert to a GeoDataFrame (Set your project's EPSG code)
geometry = [Point(xy) for xy in zip(resting_df['x'], resting_df['y'])]
points_gdf = gpd.GeoDataFrame(resting_df, geometry=geometry, crs="EPSG:27700")

# 3. Export to a GeoPackage (.gpkg) file for QGIS
# GeoPackages are preferred over Shapefiles as they support longer column names
points_gdf.to_file("qgis_resting.gpkg", layer="telemetry_fixes", driver="GPKG")
print("Spatial file successfully created for QGIS!")

In [ ]:
i_df = hmm_df[hmm_df["behaviour_state"] == "Intensive Foraging"]

# 2. Convert to a GeoDataFrame (Set your project's EPSG code)
geometry = [Point(xy) for xy in zip(i_df['x'], i_df['y'])]
points_gdf = gpd.GeoDataFrame(i_df, geometry=geometry, crs="EPSG:27700")

# 3. Export to a GeoPackage (.gpkg) file for QGIS
# GeoPackages are preferred over Shapefiles as they support longer column names
points_gdf.to_file("qgis_i.gpkg", layer="telemetry_fixes", driver="GPKG")
print("Spatial file successfully created for QGIS!")

In [ ]:
e_df = hmm_df[hmm_df["behaviour_state"] == "Extensive Foraging"]

# 2. Convert to a GeoDataFrame (Set your project's EPSG code)
geometry = [Point(xy) for xy in zip(e_df['x'], e_df['y'])]
points_gdf = gpd.GeoDataFrame(e_df, geometry=geometry, crs="EPSG:27700")

# 3. Export to a GeoPackage (.gpkg) file for QGIS
# GeoPackages are preferred over Shapefiles as they support longer column names
points_gdf.to_file("qgis_e.gpkg", layer="telemetry_fixes", driver="GPKG")
print("Spatial file successfully created for QGIS!")

In [ ]:
transit_df = hmm_df[hmm_df["behaviour_state"] == "Transit"]

# 2. Convert to a GeoDataFrame (Set your project's EPSG code)
geometry = [Point(xy) for xy in zip(transit_df['x'], transit_df['y'])]
points_gdf = gpd.GeoDataFrame(transit_df, geometry=geometry, crs="EPSG:27700")

# 3. Export to a GeoPackage (.gpkg) file for QGIS
# GeoPackages are preferred over Shapefiles as they support longer column names
points_gdf.to_file("qgis_transit.gpkg", layer="telemetry_fixes", driver="GPKG")
print("Spatial file successfully created for QGIS!")

In [ ]:
g_df = hmm_df[hmm_df["behaviour_state"].isin(["Intensive Foraging", "Extensive Foraging"])]

# 2. Convert to a GeoDataFrame (Set your project's EPSG code)
geometry = [Point(xy) for xy in zip(g_df['x'], g_df['y'])]
points_gdf = gpd.GeoDataFrame(g_df, geometry=geometry, crs="EPSG:27700")

# 3. Export to a GeoPackage (.gpkg) file for QGIS
# GeoPackages are preferred over Shapefiles as they support longer column names
points_gdf.to_file("qgis_g.gpkg", layer="telemetry_fixes", driver="GPKG")
print("Spatial file successfully created for QGIS!")

In [ ]:
# Convert latitude/longitude to projected coordinates.
hmm_df = gpd.GeoDataFrame( 
    hmm_df, 
    geometry=gpd.points_from_xy(hmm_df["Longitude"], hmm_df["Latitude"]), 
    crs="EPSG:4326"
) 
 
# Example only: British National Grid. Change if needed. 
hmm_df = hmm_df.to_crs("EPSG:27700") 
 
hmm_df["x"] = hmm_df.geometry.x 
hmm_df["y"] = hmm_df.geometry.y 

In [ ]:
habitat = gpd.read_file("NVC polygons.shp").to_crs(hmm_df.crs) 

# 1. Ensure habitat matches state_gdf's CRS before joining
site_boundary = habitat[["NVC", "geometry"]].to_crs(hmm_df.crs)

site_boundary.replace({"NVC": 'woodland'}, 'Woodland', inplace=True)

# 2. Run the spatial join using "intersects" (safest for points-in-polygons)
joined = gpd.sjoin(
    hmm_df, 
    site_boundary, 
    how="left", 
    predicate="intersects"
)

joined.replace({"NVC": 'woodland'}, 'Woodland', inplace=True)

In [ ]:
habitat_state = ( 
    joined.groupby(["NVC", "behaviour_state"]) 
    .size() 
    #.groupby(level=0) 
    #.apply(lambda x: 100 * x / x.sum()) 
    .reset_index(name="frequency") 
) 

#joined.to_csv("processed_plus_habitat.csv", index=False) 
habitat_state.to_csv("habitat_by_behaviour_state_full.csv", index=False)

In [ ]:
from shapely.geometry import Polygon

#METRIC_CRS = "EPSG:4326" 

# --- NEW: CRITICAL TOPOLOGY REPAIR STEP ---
# Check if any geometries are broken
if not site_boundary.geometry.is_valid.all():
    print("Invalid geometries found in shapefile! Repairing...")
    # .make_valid() fixes self-intersections and structural errors automatically
    site_boundary['geometry'] = site_boundary.geometry.make_valid()
# ------------------------------------------

# 2. Generate the Grid
xmin, ymin, xmax, ymax = site_boundary.total_bounds
grid_size = 5

x_coords = np.arange(xmin, xmax, grid_size)
y_coords = np.arange(ymin, ymax, grid_size)

grid_cells = []
for x in x_coords:
    for y in y_coords:
        grid_cells.append(Polygon([
            (x, y), 
            (x + grid_size, y), 
            (x + grid_size, y + grid_size), 
            (x, y + grid_size)
        ]))

grid = gpd.GeoDataFrame({'grid_id': range(len(grid_cells))}, geometry=grid_cells, crs=joined.crs)

# This clipping step will now run smoothly without any GEOS exceptions
grid = gpd.clip(grid, site_boundary)

grid.plot(column="grid_id", 
             legend=True,
             cmap="tab20b",
             figsize=(12, 10))

#grid

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon

# [Your existing code setup...]
if not site_boundary.geometry.is_valid.all():
    print("Invalid geometries found in shapefile! Repairing...")
    site_boundary['geometry'] = site_boundary.geometry.make_valid()

xmin, ymin, xmax, ymax = site_boundary.total_bounds
grid_size = 5

x_coords = np.arange(xmin, xmax, grid_size)
y_coords = np.arange(ymin, ymax, grid_size)

# MODIFIED: Create lists to store the coordinates alongside the polygons
grid_cells = []
grid_x_list = []
grid_y_list = []

for x in x_coords:
    for y in y_coords:
        grid_cells.append(Polygon([
            (x, y), 
            (x + grid_size, y), 
            (x + grid_size, y + grid_size), 
            (x, y + grid_size)
        ]))
        # Keep track of the bottom-left corner of each 5x5m cell
        grid_x_list.append(x)
        grid_y_list.append(y)

# Create the GeoDataFrame including the coordinate attributes
grid = gpd.GeoDataFrame({
    'grid_id': range(len(grid_cells)),
    'grid_x_5m': grid_x_list,
    'grid_y_5m': grid_y_list
}, geometry=grid_cells, crs=site_boundary.crs) # Use site_boundary.crs directly

# Clip the grid to the site boundary (This creates the mixed lines/polygons)
grid = gpd.clip(grid, site_boundary)

# --- FIX: Keep only Polygon and MultiPolygon geometries ---
# This safely removes any accidental LineStrings or Points caused by the clip
grid = grid[grid.geometry.type.isin(['Polygon', 'MultiPolygon'])]

# --- NEW: SAVE TO SHAPEFILE ---
# This will now export cleanly without errors!
grid.to_file("site_5m_grid.shp")

print("Grid shapefile 'site_5m_grid.shp' has been created successfully!")

In [ ]:
# Fix: Drop it again here because step 1 created it, and step 4 is another sjoin!
joined = joined.drop(columns=['index_right'], errors='ignore')

# 1. Spatial join: Only retain points that successfully hit a 20m grid cell
cattle_mapped1= gpd.sjoin(joined, grid[['grid_id', 'geometry']], how='inner', predicate='within')

# 2. Count occurrences per cell relative to valid on-site points
valid_points_total1 = len(cattle_mapped1)
cell_counts1 = cattle_mapped1['grid_id'].value_counts()
cell_o_Ac = (cell_counts1 / valid_points_total1).to_dict()

# 3. Map o_Ac back to the grid and the point dataframes
grid['o_Ac'] = grid['grid_id'].map(cell_o_Ac).fillna(0)

# Fix: Drop it again here because step 1 created it, and step 4 is another sjoin!
joined = joined.drop(columns=['index_right'], errors='ignore')

# Merge data back into your master cattle dataframe safely
joined['grid_id'] = gpd.sjoin(joined, grid[['grid_id', 'geometry']], how='left', predicate='within')['grid_id']
joined['o_Ac'] = joined['grid_id'].map(cell_o_Ac).fillna(0)

print(f"Success! Non-zero values generated: {(joined['o_Ac'] > 0).sum()} rows.")

ogrid = joined[joined["grid_id"] == 5042] 

joined

In [ ]:
# Fix: Drop the conflicting column from previous joins if it exists
#joined = joined.drop(columns=['index_right'], errors='ignore')

foraging_df = joined[joined["behaviour_state"].isin(["Intensive Foraging", "Extensive Foraging"])]

# Fix: Drop it again here because step 1 created it, and step 4 is another sjoin!
foraging_df = foraging_df.drop(columns=['index_right'], errors='ignore')

# 1. Spatial join: Only retain points that successfully hit a 20m grid cell
cattle_mapped = gpd.sjoin(foraging_df, grid[['grid_id', 'geometry']], how='inner', predicate='within')

# 2. Count occurrences per cell relative to valid on-site points
valid_points_total = len(cattle_mapped)
cell_counts = cattle_mapped['grid_id_right'].value_counts()
cell_availability = (cell_counts / valid_points_total).to_dict()

# 3. Map availability back to the grid and the point dataframes
grid['availability'] = grid['grid_id'].map(cell_availability).fillna(0)

# Fix: Drop it again here because step 1 created it, and step 4 is another sjoin!
foraging_df = foraging_df.drop(columns=['index_right'], errors='ignore')

# Merge data back into your master cattle dataframe safely
foraging_df['grid_id'] = gpd.sjoin(foraging_df, grid[['grid_id', 'geometry']], how='left', predicate='within')['grid_id_right']
foraging_df['availability'] = foraging_df['grid_id'].map(cell_availability).fillna(0)

print(f"Success! Non-zero values generated: {(foraging_df['availability'] > 0).sum()} rows.")

fgrid = foraging_df[foraging_df["grid_id"] == 5042] 

foraging_df

In [ ]:
# Spatial join: Intersect the 20m grid blocks with the underlying Habitat polygons
grid_with_habitats = gpd.sjoin(grid, site_boundary, how='inner', predicate='intersects')

# Fix: Drop it again here because step 1 created it, and step 4 is another sjoin!
grid_with_habitats = grid_with_habitats.drop(columns=['index_right'], errors='ignore')

# Aggregate actual usage (availability) vs geographic area per habitat
habitat_summary = grid_with_habitats.groupby('NVC').agg(
    total_grid_area=('geometry', lambda x: len(x) * 25), # 20x20 = 400m2 per cell
    observed_usage=('availability', 'sum')
).reset_index()

# Calculate proportions
total_area = habitat_summary['total_grid_area'].sum()
habitat_summary['expected_usage_ratio'] = habitat_summary['total_grid_area'] / total_area

# Calculate the Grazing Index: Observed Presence / Expected Baseline Area
# Index > 1 means Over-grazed; Index < 1 means Under-grazed
habitat_summary['usage_index'] = habitat_summary['observed_usage'] / habitat_summary['expected_usage_ratio']

habitat_summary

In [ ]:
# Check the coordinate ranges
print("--- GEOMETRY BOUNDS CHECK ---")
print("Grid Extents:   ", grid.total_bounds)
print("Cattle Extents: ", foraging_df.total_bounds)

# Check if ANY intersection is happening naturally
points_inside = foraging_df.within(grid.unary_union).sum()
print(f"Points physically inside the site boundary: {points_inside} out of {len(foraging_df)}")

In [ ]:
ov_av = grid_with_habitats['availability'].mean()

counts1 = grid_with_habitats.groupby('NVC').agg(
   hab_av=('availability', 'mean')
)

# Fix: Map only the 'hab_av' column from counts1
foraging_df["hab_av"] = foraging_df["NVC"].map(counts1["hab_av"])

foraging_df["GPI"] = foraging_df["hab_av"] / ov_av

foraging_df["GPI_assessment"] = np.select(
    [foraging_df["GPI"] > 1.616369, foraging_df["GPI"] <  0.917547],
    ["Over-grazed (Selected)", "Under-grazed (Avoided)"],
    default="Proportional Grazing",
)

ov = foraging_df[foraging_df["NVC"] == "Bankside veg"]

foraging_df.to_csv("foraging_glm.csv", index=False)

ov

In [ ]:
# Assigns variables to an object.
continuous1 = ["GPI"]

#Calculates descriptive statistics for numeric variables.
numeric1 = foraging_df[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = foraging_df[column].median()
print(numeric1)

In [ ]:
# o_Ac = total cell observations / total observations
# hab_av_o_Ac = total o_Ac per habitat

h_o_Ac = grid_with_habitats.groupby('NVC').agg(
   hab_av=('o_Ac', 'sum')
)

# Fix: Map only the 'hab_av' column from counts1
joined["hab_av_o_Ac"] = joined["NVC"].map(h_o_Ac["hab_av"])

ov = joined[joined["NVC"] == "pond basin"]

joined.to_csv("joined_glm.csv", index=False)

ov

In [ ]:
# Assigns variables to an object.
continuous1 = ["availability"]

#Calculates descriptive statistics for numeric variables.
numeric1 = foraging_df[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = foraging_df[column].median()
print(numeric1)

In [ ]:
# area of cells inside Habitat where av exceeds upper threshold / total area of Habitat
# Spatial join: Intersect the 20m grid blocks with the underlying Habitat polygons
grid_with_habitats1 = gpd.sjoin(grid, site_boundary, how='inner', predicate='intersects')

# Fix: Drop it again here because step 1 created it, and step 4 is another sjoin!
grid_with_habitats1 = grid_with_habitats1.drop(columns=['index_right'], errors='ignore')

hab1 = grid_with_habitats1.groupby('NVC').agg(
    total_hab_area=('geometry', lambda x: len(x) * 25), # 20x20 = 400m2 per cell
).reset_index()

foraging_df["av_q"] = np.select(
    [foraging_df["availability"] > 0.000231, foraging_df["availability"] < 0.000089],
    ["p75", "p25"],
    default="proportional",
)

p75_f_df = foraging_df[foraging_df["av_q"] == "p75"] 

grid_with_habitats2 = grid_with_habitats1[grid_with_habitats1["availability"] > 0.000231] 

hab2 = grid_with_habitats2.groupby('NVC').agg(
    total_p75_area=('geometry', lambda x: len(x) * 25),
).reset_index()

hab2["total_hab_area"] = hab2["NVC"].map(hab1["total_hab_area"])

hab2["PFF"] = hab2["total_p75_area"] / hab2["total_hab_area"]

hab2["PFF_assessment"] = np.select(
    [hab2["PFF"] > 0.331818, hab2["PFF"] < 0.016975],
    ["Over", "Under"],
    default="Proportional",
)

hab2

grid_with_habitats.to_csv("grid_glm.csv", index=False)

hab2

In [ ]:
# Assigns variables to an object.
continuous1 = ["PFF"]

#Calculates descriptive statistics for numeric variables.
numeric1 = hab2[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = hab2[column].median()
print(numeric1)

In [ ]:
hab1

In [ ]:
# needs changing manually in word then convert to csv.

gpi_pff = pd.read_csv("gpi_pff.csv")

gpi_pff["Area_"] = gpi_pff["Area"]
gpi_pff["PFF_"] = gpi_pff["PFF"]
gpi_pff["Availability_"] = gpi_pff["Availability"]
gpi_pff["GPI_"] = gpi_pff["GPI"]
gpi_pff["GPI_Assesment_"] = gpi_pff["GPI_Assesment"]

# Select only the NVC key and the specific columns you want to bring over
columns_to_add = ["NVC", "PFF_", "Availability_", "GPI_", "GPI_Assesment_"]

# Merge them all in one clean step
joined = joined.merge(gpi_pff[columns_to_add], on="NVC", how="left")

joined

In [ ]:
# Select only the NVC key and the specific columns you want to bring over
columns_to_add = ["NVC", "PFF_", "Availability_", "GPI_", "GPI_Assesment_"]

# Merge them all in one clean step
joined = joined.merge(gpi_pff[columns_to_add], on="NVC", how="left")

In [ ]:
joined.to_file("qgis_gpi_pff.gpkg", layer="telemetry_fixes", driver="GPKG")
print("Spatial file successfully created for QGIS!")

joined.columns

In [ ]:
profile = ( 
    joined.groupby(["Serial", "behaviour_state"]) #as_index=False
    .size() 
    .groupby(level=0) 
    .apply(lambda x: 100 * x / x.sum()) 
    #.reset_index(name="frequency")
) 

profile

In [ ]:
#jf = joined.loc[joined['Serial'] == 294366]
#jf = joined.loc[joined['Collar_Status'] == "OffAnimal"]
#jf = joined.loc[joined['Fence_Status'] == "NotStarted"]
jf = joined.loc[joined['hmm_state_raw'] <= 3]
#jf = jf.loc[jf['Serial'] == 294366]
#jf = jf.loc[jf['pt_numeric'] > ]
#jf = jf.loc[jf['pt_numeric'] < 500]

# 1. Initialize a clean, publication-sized figure frame
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)

habitat.plot(ax=ax, color='#f1f3f5', edgecolor='#adb5bd', linewidth=0.5)

custom_colors = {
    "Resting": "grey",            # Matte Dark Grey / Charcoal
    "Intensive Foraging": "lightgreen",   # Crisp Light Yellowish Green
    "Extensive Foraging": "darkgreen",   # Standard Publication Green (ColorBrewer standard)
    "Transit": "orange"               # High-Contrast Pop Orange
}

joined.plot(column='behaviour_state',
                 legend=True, 
                 ax=ax, 
                 cmap=None,
                 color=[custom_colors[x] for x in joined['behaviour_state']],
                 alpha=0.7,
                 markersize=0.1)

# 3. Explicitly tell matplotlib to display the legend
ax.legend(loc='lower left', frameon=True, facecolor='white', edgecolor='#e9ecef', fontsize=9)

ax.axis('off') # Deletes all surrounding axis boxes, numbers, and tick marks

plt.tight_layout()
plt.show()

In [ ]:
jf = joined.loc[joined['assigned_state'].isin([1, 4])]

fig, ax = plt.subplots(figsize=(6, 6), dpi=300)

# 1. Plot the background habitat boundaries
habitat.plot(ax=ax, color='white', edgecolor='#adb5bd', linewidth=0.5)

custom_colors = {
    "Resting": "grey",            
    "Intensive Foraging": "lightgreen",   
    "Extensive Foraging": "darkgreen",   
    "Transit": "orange"               
}

# 2. Explicitly loop and plot each behavior state as its own layer
for state, color in custom_colors.items():
    subset = jf[jf['behaviour_state'] == state]
    if not subset.empty:
        subset.plot(
            ax=ax, 
            color=color, 
            alpha=0.6, 
            markersize=0.1,  # Prevents legend icons from becoming microscopic
            label=state    # Feeds the text string directly to the legend generator
        )

# 3. Use standard matplotlib legend control (completely safe from keywords errors)
ax.legend(
    loc='lower left', 
    facecolor='white', 
    edgecolor='#e9ecef', 
    fontsize=5,
    title="Behaviour State",
    title_fontsize=5
)

ax.axis('off') 
plt.tight_layout()
plt.show()

In [ ]:
import folium
from folium.plugins import HeatMap

# next steps could include heatmap/raster weighted by grazing per habitat (as a ratio of time spent there possibly), montly or by pasture version, aand filter by fence status.

#exactcows = foraging_df.loc[cows['is_location_exact'] == 't']
jf = foraging_df.loc[foraging_df['hmm_state_raw'] == 1]

geo_data = joined[['Latitude', 'Longitude']] # Can change "cows to vairious filters from last markdown cell.

heat_data = geo_data[['Latitude', 'Longitude']].values.tolist()

map_center = [geo_data['Latitude'].mean(), geo_data['Longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=16)

HeatMap(heat_data, radius=3, blur=1, max_zoom=12).add_to(m)

m